# Observability, testing and evaluation

Assistant version 10: Gate regressions in tool trajectories, grounding and tone with a local evaluation suite.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --extra deep --extra deployment --group authoring`, then `uv run --extra deep --extra deployment --group authoring jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## LangSmith tracing; tags, metadata; redact before tracing; account-free JSON logs

Predict the output before running. The function definitions below match `examples/01_trace.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Redact before opening a trace; write a safe local JSON envelope.

Run: uv run --extra agents python modules/19_evaluation/examples/01_trace.py --offline
Expected: Email and secret fields are replaced; tracing is optional.
Cost: No model calls; an explicitly enabled live trace uploads sanitized fields. Requires: agents
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.security import log_event
from common.tracing import sanitized_trace


def main() -> None:
    settings = lesson_settings()
    # KEY IDEA: sanitization occurs before the decorated trace function receives inputs.
    safe = sanitized_trace(
        {"question": "Contact alex@example.test", "password": "synthetic"}, settings
    )
    print_text(log_event("support_demo", "demo-01", {"payload": safe}, settings))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## GenericFakeChatModel; scripted tool-call messages; fake embeddings; tool tests

Predict the output before running. The function definitions below match `examples/02_fake_models.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Separate scripted transport tests from model-quality evidence.

Run: uv run --extra agents python modules/19_evaluation/examples/02_fake_models.py --offline
Expected: A real ownership denial and two fake vectors.
Cost: No API calls; deterministic local checks. Requires: agents
"""

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model, get_embeddings
from common.pretty import print_text
from common.tools import make_order_tools


def main() -> None:
    settings = lesson_settings().model_copy(update={"course_offline": True})
    model = get_chat_model(
        settings=settings, responses=[tool_request("lookup_order", {"order_id": "BW1001"})]
    )
    call = model.invoke("Check my order.").tool_calls[0]
    # KEY IDEA: the fixture supplies the call, but the tool executes real ownership checks.
    result = make_order_tools("C002")[0].invoke(call["args"])
    assert "error" in result
    print_text(str(result))
    vectors = get_embeddings(settings=settings).embed_documents(
        ["Brightwave tent", "Brightwave pack"]
    )
    print_text(str({"vectors": len(vectors), "semantic_quality_claim": False}))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## pytest live marker; opt-in integration tests; provider/model capability matrix

Predict the output before running. The function definitions below match `examples/03_live_tests.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Expose what a provider capability matrix has and has not verified.

Run: uv run --extra agents python modules/19_evaluation/examples/03_live_tests.py --offline
Expected: A parsed company and an honest four-provider matrix.
Cost: One structured call with selected live provider; no calls offline. Requires: agents
"""

from pydantic import BaseModel

from common.errors import run_cli
from common.fixtures import structured_fixture
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text


class Greeting(BaseModel):
    company: str


def main() -> None:
    settings = lesson_settings()
    model = get_chat_model(
        settings=settings, responses=[structured_fixture(Greeting, {"company": "Brightwave"})]
    )
    result = model.with_structured_output(Greeting).invoke("Return company Brightwave.")
    # KEY IDEA: only this selected profile was invoked; adapter support is not live evidence.
    matrix = {
        name: "not tested in this run" for name in ("openai", "anthropic", "google_genai", "ollama")
    }
    matrix[settings.llm_provider] = (
        "scripted parser only" if settings.course_offline else "structured output invoked"
    )
    print_text(str({"result": result.model_dump(), "matrix": matrix}))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## LangSmith datasets and evaluate; custom evaluators; optional LLM judge

Predict the output before running. The function definitions below match `examples/04_datasets_evaluate.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Run LangSmith evaluate locally, then demonstrate a calibrated-judge boundary.

Run: uv run --extra agents python modules/19_evaluation/examples/04_datasets_evaluate.py --offline
Expected: Two local evaluator rows and an explicitly scripted judge verdict.
Cost: Up to eleven logical live calls; SDK retries may add attempts. Cloud upload is separate.
Requires: agents
"""

from common.errors import run_cli
from common.evaluation import judge_answer
from common.experiments import run_experiment
from common.lesson import lesson_settings
from common.pretty import print_text
from common.support_evaluation import load_cases


def main() -> None:
    settings = lesson_settings()
    # KEY IDEA: upload_results=False uses the real runner without creating a cloud experiment.
    print_text(str(run_experiment(load_cases()[:2], settings)))
    verdict = judge_answer(
        "What is the return window?",
        {"answer": "30 days"},
        "Unused gear within 30 days",
        "Unused gear within 30 days",
        settings,
    )
    print_text(str(verdict.model_dump()))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Tool-call sequence, arguments, ownership and forbidden-action assertions

Predict the output before running. The function definitions below match `examples/05_trajectory.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Compare tool names and arguments, including forbidden side effects.

Run: uv run --extra agents python modules/19_evaluation/examples/05_trajectory.py --offline
Expected: Expected read-only trajectory passes and a forbidden action fails.
Cost: Up to three live calls; none offline. Requires: agents
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.support_evaluation import evaluate_trajectory, support_answer


def main() -> None:
    settings = lesson_settings()
    answer = support_answer("Show BW1001", "C002", settings)
    expected = [{"name": "lookup_order", "args": {"order_id": "BW1001"}}]
    # KEY IDEA: a polite final sentence cannot excuse a forbidden tool earlier in the run.
    print_text(
        str(
            {
                "trajectory_passed": evaluate_trajectory(
                    answer["trajectory"], expected, ["send_email"]
                ),
                "ownership_denied": "error" in answer["evidence"][0],
            }
        )
    )
    assert not evaluate_trajectory([{"name": "send_email", "args": {}}], [], ["send_email"])


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Versioned prompts; held-out inputs; semantic expectations rather than exact prose

Predict the output before running. The function definitions below match `examples/06_prompt_regression.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Compare versioned prompts on held-out expectations, not exact reply text.

Run: uv run --extra agents python modules/19_evaluation/examples/06_prompt_regression.py --offline
Expected: Different prompt hashes and equal offline contract results.
Cost: Up to sixty logical calls live; SDK retries can add attempts. Zero offline. Requires: agents
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.support_evaluation import load_cases, local_evaluate


def main() -> None:
    settings = lesson_settings()
    for version in ("v1", "v2"):
        result = local_evaluate(load_cases(), settings, version=version)
        print_text(
            str(
                {
                    "version": version,
                    "passed": result["passed"],
                    "total": result["total"],
                    "prompt_hash": result["rows"][0]["answer"]["prompt_sha256"],
                }
            )
        )
    # KEY IDEA: the offline fixture ignores prompt prose, so equal scores prove no improvement.
    print_text(
        "Offline equality is expected. Compare live runs and human-reviewed failures separately."
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Local JSON evaluation; faithfulness, tone, correctness; CI thresholds and reports

Predict the output before running. The function definitions below match `examples/07_local_harness.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Report failures and demonstrate a release gate that actually rejects them.

Run: uv run --extra agents python modules/19_evaluation/examples/07_local_harness.py --offline
Expected: 6/6 offline cases pass and the strict release gate is true.
Cost: Up to thirty logical live calls; retries can add attempts. Zero offline. Requires: agents
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.release import evaluation_gate
from common.support_evaluation import load_cases, local_evaluate


def main() -> None:
    report = local_evaluate(load_cases(), lesson_settings())
    # KEY IDEA: the default quality threshold is 100%; known misses block release.
    print_text(
        str(
            {
                "passed": report["passed"],
                "total": report["total"],
                "release_allowed": evaluation_gate(report),
                "limits": report["limits"],
            }
        )
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world application entry point

Predict the output before running. The function definitions below match `use_case/support_eval.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Save a transparent local evaluation report, including answer-quality checks.

Run: uv run --extra agents python modules/19_evaluation/use_case/support_eval.py --offline
Expected: 6/6 scripted cases; offline quality gate passes; report under .runtime/.
Cost: Up to thirty logical live calls; retries can add attempts. Zero offline. Requires: agents
"""

from common.config import ROOT
from common.errors import run_cli
from common.evaluation import save_report
from common.lesson import lesson_settings
from common.pretty import print_text
from common.release import evaluation_gate
from common.support_evaluation import load_cases, local_evaluate


def main() -> None:
    settings = lesson_settings()
    report = local_evaluate(load_cases(), settings)
    report["release_allowed"] = evaluation_gate(report)
    output = ROOT / settings.runtime_dir / "support-evaluation.json"
    save_report(report, output)
    print_text(
        str(
            {
                "passed": report["passed"],
                "total": report["total"],
                "release_allowed": report["release_allowed"],
                "report": str(output),
            }
        )
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run --extra apps pytest modules/19_evaluation/tests` for the supplied solution checks.